# Loading data

Downloads CelebA images from HuggingFace balanced on `CONCEPT`, saves to a shared pool.

| path | content |
|---|---|
| `data/images/` | shared image pool (224×224 JPG) |
| `data/metadata.csv` | filename, split + all saved attributes |

**Change only `CONCEPT`** — everything else follows automatically.  
For downloading all concepts at once, use `scripts/download_data.ipynb` instead.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from dotenv import load_dotenv
from tqdm import tqdm

warnings.filterwarnings('ignore')
load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CONCEPT = 'eyeglasses'   # change to any key in ATTR_MAP below
DS_SIZE = 256

# HuggingFace attribute name → CSV column name
ATTR_MAP = {
    'Eyeglasses':   'eyeglasses',
    'Bald':         'bald',
    'Chubby':       'chubby',
    'Wearing_Hat':  'wearing_hat',
    'Male':         'male',
    'Smiling':      'smiling',
    'Straight_Hair':'straight_hair',
    'Wavy_Hair':    'wavy_hair',
    'Big_Nose':     'big_nose',
    'Blond_Hair':   'blond_hair',
    'Young':        'young',
}
_COL_TO_HF = {v: k for k, v in ATTR_MAP.items()}

assert CONCEPT in ATTR_MAP.values(), (
    f"'{CONCEPT}' not in ATTR_MAP. Available: {list(ATTR_MAP.values())}"
)
BALANCE_HF_ATTR = _COL_TO_HF[CONCEPT]

TRAIN_N = 7 * DS_SIZE   # 1792 → 896 pos + 896 neg
TEST_N  = 1 * DS_SIZE   #  256 → 128 pos + 128 neg

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent

IMAGES_DIR    = ROOT / 'data' / 'images'
METADATA_PATH = ROOT / 'data' / 'metadata.csv'
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Concept          : {CONCEPT}  (HF attr: {BALANCE_HF_ATTR})')
print(f'Train target     : {TRAIN_N} ({TRAIN_N//2} pos + {TRAIN_N//2} neg)')
print(f'Test  target     : {TEST_N}  ({TEST_N//2}  pos + {TEST_N//2}  neg)')
print(f'Images dir       : {IMAGES_DIR}')
print(f'Metadata path    : {METADATA_PATH}')

In [ ]:
def collect_split(hf_split, split_label, n, start_idx, metadata_list):
    ds = load_dataset('flwrlabs/celeba', split=hf_split, streaming=True, token=HF_TOKEN)
    target = n // 2
    pos_count = neg_count = 0
    idx = start_idx

    print(f'\n[{split_label}] target: {target} pos + {target} neg')
    for item in tqdm(ds, desc=split_label):
        is_pos = bool(item[BALANCE_HF_ATTR])
        if is_pos and pos_count >= target:
            continue
        if not is_pos and neg_count >= target:
            continue

        img = item['image'].convert('RGB').resize((224, 224))
        filename = f'img_{idx:06d}.jpg'
        img.save(IMAGES_DIR / filename, quality=90)

        row = {'filename': filename, 'split': split_label}
        for hf_attr, col in ATTR_MAP.items():
            row[col] = int(item[hf_attr]) if hf_attr in item else None
        metadata_list.append(row)

        if is_pos:
            pos_count += 1
        else:
            neg_count += 1
        idx += 1

        if pos_count >= target and neg_count >= target:
            break

    print(f'  -> collected {pos_count} pos + {neg_count} neg = {idx - start_idx} images')
    return idx

In [ ]:
if METADATA_PATH.exists():
    df_existing = pd.read_csv(METADATA_PATH)
    if CONCEPT in df_existing.columns:
        print(f'metadata.csv already exists with column "{CONCEPT}" — skipping download.')
        print(f'Delete {METADATA_PATH} to re-download.')
    else:
        print(f'metadata.csv exists but missing column "{CONCEPT}".')
        print('Use scripts/download_data.ipynb to add missing concepts.')
else:
    metadata_list = []
    idx = 0
    idx = collect_split('train', 'train', TRAIN_N, idx, metadata_list)
    idx = collect_split('test',  'test',  TEST_N,  idx, metadata_list)

    df = pd.DataFrame(metadata_list)
    col_order = ['filename', 'split'] + list(ATTR_MAP.values())
    df = df[[c for c in col_order if c in df.columns]]
    df.to_csv(METADATA_PATH, index=False)
    print(f'\nSaved {len(df)} records → {METADATA_PATH}')

In [ ]:
df = pd.read_csv(METADATA_PATH)
print(f'Total images : {len(df)}')
print(df['split'].value_counts().to_string())
if CONCEPT in df.columns:
    print(f'\n{CONCEPT} balance per split:')
    print(df.groupby('split')[CONCEPT].mean().mul(100).round(1).to_string())